In [2]:
import torch 
import torch.nn as nn
import numpy as np

device = torch.device('cuda')
print(f'using {device}')

using cuda


In [3]:
class PINNs(nn.Module):

    NUM_NEUROS = 40
    NUM_LAYERS = 9

    def __init__(self):
        super().__init__()
        
        layers = []

        #first_layer
        layers.append(nn.Linear(2, self.NUM_NEUROS))
        layers.append(nn.Tanh())

        # black box of 8 layers
        for _ in range(self.NUM_LAYERS - 1):
            layers.append(nn.Linear(self.NUM_NEUROS, self.NUM_NEUROS))
            layers.append(nn.Tanh())

        layers.append(nn.Linear(self.NUM_NEUROS, 2))

        self.network = nn.Sequential(*layers)
        

    def forward(self, t, x):
        inputs = torch.cat([t, x], dim=1)
        outputs = self.network(inputs)
        u = outputs[:,0:1]
        v = outputs[:,1:2]
        return u, v

In [4]:
def physics_residual(model, t, x):
    t = t.clone().detach().requires_grad_(True)
    x = x.clone().detach().requires_grad_(True)

    u, v= model(t, x)

    u_t = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True
    )[0]
    u_x = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True
    )[0]
    u_xx = torch.autograd.grad(
        u_x, x,
        grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True
    )[0]


    v_t = torch.autograd.grad(
        v, t,
        grad_outputs=torch.ones_like(v), create_graph=True, retain_graph=True
    )[0]
    v_x = torch.autograd.grad(
        v, x,
        grad_outputs=torch.ones_like(v), create_graph=True, retain_graph=True
    )[0]
    v_xx = torch.autograd.grad(
        v_x, x,
        grad_outputs=torch.ones_like(v), create_graph=True, retain_graph=True
    )[0]

    h_sq = u**2 + v**2

    f_u = -v_t + 0.5 * u_xx + h_sq * u
    f_v = u_t + 0.5 * v_xx + h_sq * v

    return f_u, f_v

In [5]:
def boundary_residual(model, t_b):
    """Compute periodic boundary mismatch at x=-5 and x=5"""
    t_b = t_b.clone().detach().requires_grad_(True)
    x_left = (-5.0 * torch.ones_like(t_b)).requires_grad_(True)
    x_right = (5.0 * torch.ones_like(t_b)).requires_grad_(True)

    # Evaluate at left boundary
    u_L, v_L = model(t_b, x_left)
    u_x_L = torch.autograd.grad(u_L, x_left, grad_outputs=torch.ones_like(u_L),
                                  create_graph=True, retain_graph=True)[0]
    v_x_L = torch.autograd.grad(v_L, x_left, grad_outputs=torch.ones_like(v_L),
                                  create_graph=True, retain_graph=True)[0]

    # Evaluate at right boundary
    u_R, v_R = model(t_b, x_right)
    u_x_R = torch.autograd.grad(u_R, x_right, grad_outputs=torch.ones_like(u_R),
                                  create_graph=True, retain_graph=True)[0]
    v_x_R = torch.autograd.grad(v_R, x_right, grad_outputs=torch.ones_like(v_R),
                                  create_graph=True, retain_graph=True)[0]

    return u_L, v_L, u_R, v_R, u_x_L, v_x_L, u_x_R, v_x_R

In [6]:
def compute_loss(model, t_ic, x_ic, u_ic, v_ic, t_b, t_f, x_f):
    
    u_pred, v_pred = model(t_ic, x_ic)
    mse_a = torch.mean((u_pred - u_ic)**2 + (v_pred - v_ic)**2)

    u_L, v_L, u_R, v_R, u_x_L, v_x_L, u_x_R, v_x_R = boundary_residual(model, t_b)
    mse_b = torch.mean(
        (u_L - u_R)**2 + (v_L - v_R)**2 +
        (u_x_L - u_x_R)**2 + (v_x_L - v_x_R)**2
    )

    f_u, f_v = physics_residual(model, t_f, x_f)
    mse_f = torch.mean(f_u**2 + f_v**2)

    loss = mse_a + mse_b + mse_f
    return loss, mse_a, mse_b, mse_f

In [7]:
# Number of points
N_ic = 50    # initial condition points
N_bc = 50    # boundary condition points (total, both sides)
N_f = 20000   # c50ocation points

# ---- Initial condition: t=0, x random in [-5,5] ----
# since the solution has 2 parts, real and imaginary, u, v
x_ic = torch.linspace(-5, 5, N_ic).unsqueeze(1)
t_ic = torch.zeros(N_ic, 1)
u_ic = 2 * 1 / torch.cosh(x_ic)
v_ic = torch.zeros(N_ic, 1)

# ---- Boundary conditions ----
# Left boundary: x=0, t random in [-1,1]
t_bc = torch.rand(N_bc, 1) * ( torch.pi/2 )

from scipy.stats import qmc

sampler = qmc.LatinHypercube(d=2)
sample = sampler.random(N_f)
# ---- Collocation points: random in interior ----
t_f = torch.tensor(sample[:,0:1] * np.pi/2, dtype=torch.float32)
x_f = torch.tensor(sample[:,1:2] * 10 - 5, dtype=torch.float32)

x_ic = x_ic.to(device)
t_ic = t_ic.to(device)
u_ic = u_ic.to(device)
v_ic = v_ic.to(device)
t_bc = t_bc.to(device)
t_f = t_f.to(device)
x_f = x_f.to(device)

In [8]:
# ============================================
# 6. Training
# ============================================
model = PINNs().to(device)

# Phase 1: Adam
optimizer_adam = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Phase 1: Adam")
for epoch in range(5000):
    optimizer_adam.zero_grad()
    loss, mse_a, mse_b, mse_f = compute_loss(
        model, t_ic, x_ic, u_ic, v_ic, t_bc, t_f, x_f
    )
    loss.backward()
    optimizer_adam.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch:5d} | Loss: {loss.item():.6f} | "
              f"MSE_a: {mse_a.item():.6f} | "
              f"MSE_b: {mse_b.item():.6f} | "
              f"MSE_f: {mse_f.item():.6f}")

# Phase 2: L-BFGS
print("\nPhase 2: L-BFGS")
optimizer_lbfgs = torch.optim.LBFGS(
    model.parameters(),
    lr=1.0,
    max_iter=50000,
    max_eval=50000,
    tolerance_grad=1e-7,
    tolerance_change=1e-9,
    history_size=50,
    line_search_fn='strong_wolfe'
)

iteration = [0]
def closure():
    optimizer_lbfgs.zero_grad()
    loss, mse_a, mse_b, mse_f = compute_loss(
        model, t_ic, x_ic, u_ic, v_ic, t_bc, t_f, x_f
    )
    loss.backward()

    iteration[0] += 1
    if iteration[0] % 500 == 0:
        print(f"L-BFGS iter {iteration[0]:5d} | Loss: {loss.item():.6f} | "
              f"MSE_a: {mse_a.item():.6f} | "
              f"MSE_b: {mse_b.item():.6f} | "
              f"MSE_f: {mse_f.item():.6f}")
    return loss

optimizer_lbfgs.step(closure)

Phase 1: Adam
Epoch     0 | Loss: 0.759276 | MSE_a: 0.759256 | MSE_b: 0.000017 | MSE_f: 0.000003
Epoch   500 | Loss: 0.052522 | MSE_a: 0.024590 | MSE_b: 0.000204 | MSE_f: 0.027728
Epoch  1000 | Loss: 0.078256 | MSE_a: 0.028819 | MSE_b: 0.028047 | MSE_f: 0.021390
Epoch  1500 | Loss: 0.035497 | MSE_a: 0.016648 | MSE_b: 0.000188 | MSE_f: 0.018661
Epoch  2000 | Loss: 0.030731 | MSE_a: 0.014455 | MSE_b: 0.000184 | MSE_f: 0.016091
Epoch  2500 | Loss: 0.028603 | MSE_a: 0.014547 | MSE_b: 0.000230 | MSE_f: 0.013827
Epoch  3000 | Loss: 0.025304 | MSE_a: 0.012214 | MSE_b: 0.000223 | MSE_f: 0.012867
Epoch  3500 | Loss: 0.022794 | MSE_a: 0.010874 | MSE_b: 0.000121 | MSE_f: 0.011799
Epoch  4000 | Loss: 0.049529 | MSE_a: 0.010941 | MSE_b: 0.005768 | MSE_f: 0.032821
Epoch  4500 | Loss: 0.018727 | MSE_a: 0.009027 | MSE_b: 0.000098 | MSE_f: 0.009602

Phase 2: L-BFGS
L-BFGS iter   500 | Loss: 0.004370 | MSE_a: 0.001312 | MSE_b: 0.000027 | MSE_f: 0.003031
L-BFGS iter  1000 | Loss: 0.001339 | MSE_a: 0.0002

tensor(0.0202, device='cuda:0', grad_fn=<AddBackward0>)